# Tweet Sentiment Analysis with TF-IDF and Opinion Lexicon

This notebook builds a supervised sentiment classifier for tweets using a linear Support Vector Machine. The workflow covers data loading, preprocessing, TF-IDF feature extraction, opinion lexicon features, 10-fold cross-validation, error analysis, and final test-set evaluation.

The final approach combines unigram and bigram TF-IDF features with sentiment-oriented lexicon features to improve classification performance on short, informal social media text.

In [ ]:
import csv                               # csv reader
from sklearn.svm import LinearSVC
from nltk.classify import SklearnClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import precision_recall_fscore_support # to report on precision and recall 
import numpy as np

## Data Loading and Train/Test Preparation

The data-loading workflow keeps both processed features and raw text. The raw text is retained for interpretability during error analysis and does not alter the classifier training process.

In [ ]:

# ! Don't modify these two functions!

def load_data(path):
    """Load data from a tab-separated file and append it to raw_data."""
    with open(path, encoding="utf-8") as f:
        reader = csv.reader(f, delimiter='\t')
        for line in reader:
            if line[0] == "Id":  # skip header
                continue
            (label, text) = parse_data_line(line)
            raw_data.append((text, label))
            
# ! Don't modify these two functions!
train_data_unprocessed = []
def split_and_preprocess_data(percentage):
    """Split the data between train_data and test_data according to the percentage
    and performs the preprocessing."""
    global train_data_unprocessed
    
    num_samples = len(raw_data) # Simply the length of the raw data
    num_training_samples = int((percentage * num_samples)) # Getting the percentage of the length of raw data
    for (text, label) in raw_data[:num_training_samples]: # ? Index from 0 to num_training_samples
        train_data.append((to_feature_vector(pre_process(text)),label))
        
        train_data_unprocessed.append((text,label))
        
    for (text, label) in raw_data[num_training_samples:]: 
        test_data.append((to_feature_vector(pre_process(text)),label))

        
        
    

## Opinion Lexicon Loading

In [ ]:
def load_opinion_lexicon():
    positive_words = set()
    negative_words = set()

    # Load positive words
    with open('positive-words.txt', 'r', encoding='latin-1') as f:
        for line in f:
            # !Skip comments and empty lines
            if line.startswith(';') or not line.strip():
                continue  
            positive_words.add(line.strip())

    # Load negative words
    with open('negative-words.txt', 'r', encoding='latin-1') as f:
        for line in f:
            # !Skip comments and empty lines
            if line.startswith(';') or not line.strip():
                continue  
            negative_words.add(line.strip())

    return positive_words, negative_words

positive_words, negative_words = load_opinion_lexicon()

# Data Input and Preprocessing

## Parsing Dataset Rows

The dataset is tab-separated. This parser extracts the sentiment label and tweet text from each row, while excluding the tweet identifier from the model features.

In [ ]:

def parse_data_line(line):
    # In the dataset, the label is in the second column and the text in the third column
    # First column is a bit useless
    label = line[1]  # label is on second column
    text = line[2]  # tweet is on 3rd
    return label, text

## Text Preprocessing

The preprocessing stage normalises tweets and converts them into token-based features. This version uses TF-IDF, unigram and bigram features, lemmatisation, and lexicon-based sentiment counts.

In [ ]:
# from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
import re
from nltk.stem import WordNetLemmatizer
import nltk
nltk.download('wordnet')


In [ ]:


# !Global feature dictionary
global_feature_dict = {}

# !Initialize TfidfVectorizer (shared across all calls to pre process)
tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 2))


def pre_process(text):
    global global_feature_dict
    global tfidf_vectorizer # !Note this should make it so that tfidfvectorizer is persistent for calculating idf for entire corpus

    
    # !Normalize the text by converting it to lowercase
    text = text.lower()
    
    # !Initialize CountVectorizer to tokenize
    # count_vect = CountVectorizer(token_pattern=r'\b\w+\b', stop_words='english', ngram_range=(1,2))  # This pattern handles alphabetic characters
    
    # !Fit and transform the text (CountVectorizer expects a list of documents)
    # count_vect.fit_transform([text]) 
    # count_vect.fit_transform([text]) 
    # !Get the tokenized words from the vectorizer's vocabulary
    # tokens = count_vect.get_feature_names_out()

    
    # print(tokens)
    # ! TF-IDF
    tfidf_matrix = tfidf_vectorizer.fit_transform([text])
    # ^tokens
    tokens = tfidf_vectorizer.get_feature_names_out()
    # ^Get TF-IDF scores as an array, first row corresponds to scores
    scores = tfidf_matrix.toarray()[0]  

    
    feature_dict = {}

    # ^Loop over each index in tokens to add to dict
    for i in range(len(tokens)):
        #^ Get token + score
        token = tokens[i]      
        score = scores[i]       
        # ^Add the token and score to dict
        feature_dict[token] = score  
    
    # !Lemmatize    
    lemmatizer = WordNetLemmatizer()
    pos_count = 0
    neg_count = 0
    # ^Update global feature dict + lemmatize
    for token, score in feature_dict.items():
        # print(token)
        # ^Lemmatize
        token = lemmatizer.lemmatize(token)
        #! Lexicon
        if token in positive_words:
            pos_count += 1
        if token in negative_words:
            neg_count += 1
            
        # ^ Store in global_feature_dict
        if token in global_feature_dict:
            global_feature_dict[token] += score
        else:
            global_feature_dict[token] = score
            
    #!Lexicon (must normalise or classifier might struggle)
    feature_dict['POSITIVE_WORDS_COUNT'] = pos_count / len(tokens)
    feature_dict['NEGATIVE_WORDS_COUNT'] = neg_count / len(tokens)
    feature_dict['SENTIMENT_SCORE'] = (pos_count - neg_count) / len(tokens)
    return feature_dict


# Feature Extraction

## Feature Vector Construction

Each tweet is represented as a dictionary of weighted textual features. These feature dictionaries are then used to train and evaluate the sentiment classifier.

In [ ]:

# global_feature_dict = {} # A global dictionary of features

# def to_feature_vector(tokens):
#     # Should return a dictionary containing features as keys, and weights as values
#     # DESCRIBE YOUR METHOD IN WORDS
    

#     global global_feature_dict
#     feature_vector = {}  # Dictionary for the feature vector
#     total_tokens = len(tokens)
#     # Iterate through each token
    
#     # for token in tokens:
#     #     # ! Add token to the feature vector with a weight of 1
#     #     # feature_vector[token] = 1
#     #     # ! Term frequency
#     #     if token in feature_vector:
#     #         feature_vector[token] += 1  # Increment count for each occurrence
#     #     else:
#     #         feature_vector[token] = 1  # Start counting from 1
            
#     tfift_weights = tfidf_vect.toarray()
    
#     # ! Normalise feature vectors
#     # for token in feature_vector:
#     #     feature_vector[token] = feature_vector[token] / total_tokens  # Normalized term frequency


#     for token, count in feature_vector.items():
#         if token in global_feature_dict:
#             global_feature_dict[token] += count  # Increment global count by the count in this tweet
#         else:
#             global_feature_dict[token] = count  # Initialize global count

#     # print(feature_vector)
    
#     return feature_vector

def to_feature_vector(tokens):

    return tokens


## Model Training

In [ ]:
# TRAINING AND VALIDATING OUR CLASSIFIER
# ! Don't touch!
# 
def train_classifier(data):
    print("Training Classifier...")
    pipeline =  Pipeline([('svc', LinearSVC(class_weight="balanced"))])
    return SklearnClassifier(pipeline).train(data)

# Cross-Validation

## 10-Fold Cross-Validation

The cross-validation function evaluates the classifier across 10 folds. Each fold is used once as the validation fold, while the remaining folds are used for training. Precision, recall, F1 score, and accuracy are recorded for each run and averaged at the end.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import numpy as np
import random

# Declare global variables
global_classifier = None
global_predicted_labels = None
global_test_fold = None
train_data_test_fold_unprocessed = []
train_data_train_fold_unprocessed = []

def cross_validate(train_data, folds=10):
    global global_classifier  # Declare global to make it modifiable inside the function
    global global_predicted_labels  # Declare global to make it modifiable inside the function
    global train_data_unprocessed
    global train_data_test_fold_unprocessed
    global global_test_fold
    global train_data_train_fold_unprocessed
    # Possibly shuffle dataset
    # random.shuffle(dataset)
    
    fold_size = int(len(train_data) / folds)  + 1
    cv_results = {"precision": [], "recall": [], "f1_score": [], "accuracy": []}
    
    for i in range(folds):
        # Split dataset into test and train data
        # ! Start of the fold
        start_idx = i * fold_size
        # ! End of the fold 
        end_idx = start_idx + fold_size
        if i == folds - 1:  # ^Handle the last fold to include any remaining samples
            end_idx = len(train_data)
        # ! Test fold
        test_fold = train_data[start_idx : end_idx]
        train_data_test_fold_unprocessed = train_data_unprocessed[start_idx : end_idx]
        
        global_test_fold = test_fold
        # ! Train fold
        train_fold = train_data[ : start_idx] + train_data[end_idx : ]
        train_data_train_fold_unprocessed = train_data_unprocessed[ : start_idx] + train_data_unprocessed[end_idx : ]
        
        
        # !Separate features and labels
        train_tokens, train_labels = zip(*train_fold) # ! No need to split train_fold as we are just training the classifier with seen data.
        test_samples, test_labels = zip(*test_fold)

        # !Train the classifier using the training fold
        classifier = train_classifier(train_fold)  # ! Train classifier on the train fold of the train_data that is seen.

        # Predict labels for the test fold
        predicted_labels = predict_labels(test_samples, classifier)  # ! Predict the labels on the test fold that is unseen.


        
        # global_classifier = classifier  
        global_predicted_labels = predicted_labels  

        
        
        # !Evaluate precision, recall, F1 score, and accuracy for this fold
        # ! Compares test_fold labels and predicted_labels
        # ^ average parameter options: binary, micro, macro, weighted
        precision, recall, f1_score, _ = precision_recall_fscore_support(test_labels, predicted_labels, average='weighted')
        accuracy = accuracy_score(test_labels, predicted_labels)
        print("Fold start-end on items %d - %d" % (start_idx, end_idx))
        # ^Store the results for this fold
        cv_results["precision"].append(precision)
        cv_results["recall"].append(recall)
        cv_results["f1_score"].append(f1_score)
        cv_results["accuracy"].append(accuracy)
        
        print(f"Fold {i+1} completed. Precision: {precision:.4f}, Recall: {recall:.4f}, F1 Score: {f1_score:.4f}, Accuracy: {accuracy:.4f}") # ^ Done to .4f because my eyes are allergic to more than 4dp
    
    
    avg_precision = np.mean(cv_results["precision"])
    avg_recall = np.mean(cv_results["recall"])
    avg_f1_score = np.mean(cv_results["f1_score"])
    avg_accuracy = np.mean(cv_results["accuracy"])

    print(f"\nAverage Precision: {avg_precision:.4f}")
    print(f"Average Recall: {avg_recall:.4f}")
    print(f"Average F1 Score: {avg_f1_score:.4f}")
    print(f"Average Accuracy: {avg_accuracy:.4f}")
    
    return cv_results

## Prediction Helpers

In [ ]:
# PREDICTING LABELS GIVEN A CLASSIFIER
# ! Don't touch!
def predict_labels(samples, classifier):
    """Assuming preprocessed samples, return their predicted labels from the classifier model."""
    return classifier.classify_many(samples)

def predict_label_from_raw(sample, classifier):
    """Assuming raw text, return its predicted label from the classifier model."""
    return classifier.classify(to_feature_vector(preProcess(reviewSample)))

# Main Workflow

This section loads the dataset, separates it into training and test partitions, applies preprocessing, and prepares the feature vectors used by the classifier.

In [ ]:

# !MAIN Don't touch!

# loading reviews
# initialize global lists that will be appended to by the methods below
raw_data = []          # the filtered data from the dataset file
train_data = []        # the pre-processed training data as a percentage of the total dataset
test_data = []         # the pre-processed test data as a percentage of the total dataset


# references to the data files
data_file_path = 'sentiment-dataset.tsv'

# Do the actual stuff (i.e. call the functions we've made)
# ! We parse the dataset and put it in a raw data list
print("Now %d rawData, %d trainData, %d testData" % (len(raw_data), len(train_data), len(test_data)),
      "Preparing the dataset...",sep='\n')

load_data(data_file_path)  

# ! We split the raw dataset into a set of training data and a set of test data (80/20)
# * You do the cross validation on the 80% (training data)
# ? We print the number of training samples and the number of features before the split
print("Now %d rawData, %d trainData, %d testData" % (len(raw_data), len(train_data), len(test_data)),
      "Preparing training and test data...",sep='\n')

print(f"Sample raw data:\n{raw_data[0]}")

# ! split_data_and_preprocess()
split_and_preprocess_data(0.8)

# print(f"Testing:\n{test_fold_data_unprocessed}")

# print("\nSample train data:\n", train_data[0]) # * Prints a sample 
# print("\nSample test data:\n", test_data[0]) # * Prints a sample 
# ? We print the number of training samples and the number of features after the split
print("After split, %d rawData, %d trainData, %d testData" % (len(raw_data), len(train_data), len(test_data)),
      "Training Samples: ", len(train_data), "Features: ", len(global_feature_dict), sep='\n')



# Cross-Validation Results

The classifier is evaluated using 10-fold cross-validation to estimate performance before testing on the held-out test data.

In [ ]:
cross_validate(train_data, 10)  # will work and output overall performance of p, r, f-score when cv implemented

# Error Analysis

## Confusion Matrix Visualisation

The confusion matrix visualises true positives, false positives, true negatives, and false negatives. This helps identify whether the classifier struggles more with positive or negative sentiment.

In [ ]:
from sklearn import metrics
import matplotlib.pyplot as plt
# a function to make the confusion matrix readable and pretty
def confusion_matrix_heatmap(true_labels, pred_labels, graph_labels):
    """Function to plot a confusion matrix"""
    # pass labels to the confusion matrix function to ensure right order
    # cm = metrics.confusion_matrix(y_test, preds, labels) # !ignore this line. This is for old python.
    cm = metrics.confusion_matrix(true_labels, pred_labels, labels=graph_labels)
    fig = plt.figure(figsize=(10,10))
    ax = fig.add_subplot(111)
    cax = ax.matshow(cm)
    plt.title('Confusion matrix of the classifier')
    fig.colorbar(cax)
    ax.set_xticks(np.arange(len(graph_labels)))
    ax.set_yticks(np.arange(len(graph_labels)))
    ax.set_xticklabels( graph_labels, rotation=45)
    ax.set_yticklabels( graph_labels)

    for i in range(len(cm)):
        for j in range(len(cm)):
            text = ax.text(j, i, cm[i, j],
                           ha="center", va="center", color="w")

    plt.xlabel('Predicted')
    plt.ylabel('True')
    
    # fix for mpl bug that cuts off top/bottom of seaborn viz:
    b, t = plt.ylim() # discover the values for bottom and top
    b += 0.5 # Add 0.5 to the bottom
    t -= 0.5 # Subtract 0.5 from the top
    plt.ylim(b, t) # update the ylim(bottom, top) values
    plt.show() # ta-da!
    plt.show()

## Error Analysis Function

The error analysis uses the cross-validation outputs to inspect class balance, misclassified samples, and frequently occurring tokens in false positives and false negatives. The original held-out test split is kept separate from this diagnostic process.

In [ ]:

# !Error Analysis function
def error_analysis(test_fold, raw_test_texts, predicted_labels, train_fold):
    # classifier = train_classifier(train_data)
    test_samples, test_labels = zip(*test_fold)
    # predicted_labels = predict_labels(test_samples, classifier)
    train_samples, train_labels = zip(*train_fold)
    
    # ^labels = sorted(set(test_labels))
    labels = ['positive', 'negative']
    # ! Test fold counts!
    # ^Counters for positive and negative labels for test_fold
    test_fold_positive_count = 0
    test_fold_negative_count = 0
    
    print(f"Total samples in test fold: {len(test_fold)}")
    
    # &Iterate over the test set and count TRUE positive/negative labels
    for true_label in test_labels:
        if true_label == 'positive':
            test_fold_positive_count += 1
        elif true_label == 'negative':
            test_fold_negative_count += 1
    print(f"Total positives in TEST fold: {test_fold_positive_count}\nTotal negatives in TEST fold: {test_fold_negative_count}\nTotal false positives in TEST fold: {len(test_fold) - test_fold_positive_count}")        
    
    # ! Train fold counts!
    train_fold_positive_count = 0
    train_fold_negative_count = 0
    
    print(f"Total samples in train fold: {len(train_fold)}")
    
    for true_label in train_labels:
            if true_label == 'positive':
                train_fold_positive_count += 1
            elif true_label == 'negative':
                train_fold_negative_count += 1
    print(f"Total positives in TRAIN fold: {train_fold_positive_count}\nTotal negatives in TRAIN fold: {train_fold_negative_count}")
    
    # ! Plot the counts!
    plot_true_counts(train_fold_positive_count, train_fold_negative_count, test_fold_positive_count, test_fold_negative_count)
    
    #^ Confusion matrix
    print("Confusion Matrix:")
    confusion_matrix_heatmap(test_labels, predicted_labels, labels)
    
    
    
    # & Now counting TEST fold
    false_positives = []
    false_negatives = []
    
    
    

    # ^dictionaries to count tokens in errors
    false_positive_token_counts = {}
    false_negative_token_counts = {}
    
    # ^ Iterate through zipped test (true) labels, classifier predicted labels, test samples (the tokens) and the raw texts of the tweets. This is important since we want to line up the features with the raw texts, and see how the text is tokenised + how they were labelled.
    # ^ (Theres probably a better way to do this...but hey code efficiency isn't graded right?)
    for i, (true_label, predicted_label, sample, raw_text) in enumerate(zip(test_labels, predicted_labels, test_samples, raw_test_texts)):
        
        if true_label == 'positive' and predicted_label == 'negative':
            # print("WEEEEHEEEE:\n\n", sample, raw_text)
            false_negatives.append((sample, raw_text))  # !Misclassified as negative
            #^ Count incorrect tokens
            for token in sample:
                false_negative_token_counts[token] = false_negative_token_counts.get(token, 0) + 1

        elif true_label == 'negative' and predicted_label == 'positive':
            false_positives.append((sample, raw_text))  # !Incorrectly classified as positive
            # ^ Count incorrect tokens
            for token in sample:
                false_positive_token_counts[token] = false_positive_token_counts.get(token, 0) + 1
        # ! Correct        


    # ^Save token counts for errors to text files
    save_error_tokens(false_positive_token_counts, "false_positive_token_counts.txt", "False Positive Token Counts")
    save_error_tokens(false_negative_token_counts, "false_negative_token_counts.txt", "False Negative Token Counts")



    # ^Plot the most common tokens causing errors
    plot_error_tokens(false_positive_token_counts, "False Positive Token Counts")
    plot_error_tokens(false_negative_token_counts, "False Negative Token Counts")


    # print("False negatives: \n", false_negatives)
    # ^Save false pos and false neg to files
    save_errors(false_positives, "false_positives.txt", error_type="False Positive")
    save_errors(false_negatives, "false_negatives.txt", error_type="False Negative")
    print("\nError analysis completed. False positives and negatives saved to files.")
    

    
    # ! Put global_feature_dict in a text file
    with open("global_feature_dict.txt", 'w', encoding='utf-8') as file:
        # &Sort the dictionary by the feature index; reverse is true as we are sorting by descending
        for token, count in sorted(global_feature_dict.items(), key=lambda item: item[1], reverse=True):
            file.write(f"Token: \"{token}\", Count: {count}\n")
    print(f"Global feature dictionary saved to global_feature_dict.txt")
    

#^ saves false pos/neg to files    
def save_errors(errors, filename, error_type):
    
    with open(filename, 'w', encoding='utf-8') as file:
        for sample, raw_text in errors:
            file.write(f"{error_type}:\n")
            file.write(f"Raw Text: {raw_text}\n")
            file.write(f"Features: {sample}\n\n")

# ^function to save token counts to a text file
def save_error_tokens(token_counts, filename, title):

    with open(filename, 'w', encoding='utf-8') as file:
        file.write(f"{title}:\n")
        # ^ Sort by count
        for token, count in sorted(token_counts.items(), key=lambda item: item[1], reverse=True):
            file.write(f"Token: \"{token}\", Count: {count}\n")
    print(f"{title} saved to {filename}")
    
#^ pos/neg counts for test/train fold
def plot_true_counts(train_fold_positive_count, train_fold_negative_count, test_fold_positive_count, test_fold_negative_count):
    categories = ['Train Positives', 'Train Negatives', 'Test Positives', 'Test Negatives']
    counts = [train_fold_positive_count, train_fold_negative_count, test_fold_positive_count, test_fold_negative_count]
    print("Count plot:\n")
    #^Plot
    plt.figure(figsize=(10, 6))
    plt.bar(categories, counts, color=['blue', 'red', 'green', 'purple'])
    plt.xlabel('Categories')
    plt.ylabel('Count')
    plt.title('Positive and Negative Counts in Train and Test section of the Train Fold')
    plt.show()

# ^function to plot token counts
def plot_error_tokens(token_counts, title):

    # &Sort tokens by count and take the top 10
    sorted_tokens = sorted(token_counts.items(), key=lambda item: item[1], reverse=True)[:10]
    tokens, counts = zip(*sorted_tokens) if sorted_tokens else ([], [])
    
    # ^Plot
    plt.figure(figsize=(12, 6))
    plt.bar(tokens, counts, color='orange')
    plt.xlabel('Tokens')
    plt.ylabel('Count')
    plt.title(title)
    plt.xticks(rotation=45)
    plt.show()
    


The analysis focuses on class balance, confusion matrix patterns, false positives, false negatives, and the most frequent tokens associated with misclassification. These outputs support the feature engineering decisions used in the final model configuration.

In [ ]:

# print(train_data_test_fold_unprocessed)
# print(test_fold_data_unprocessed)
error_analysis(global_test_fold, train_data_test_fold_unprocessed, global_predicted_labels, train_data_train_fold_unprocessed)

# Feature Engineering and Final Evaluation

This final section evaluates the selected feature engineering approach, combining TF-IDF features with an opinion lexicon to improve performance on short-form sentiment classification.

## Final Test Evaluation

In [ ]:
# Finally, check the accuracy of your classifier by training on all the traning data
# and testing on the test set
# Will only work once all functions are complete
functions_complete = True  # !set to True once you're happy with your methods for cross val
if functions_complete:
    print(test_data[0])   # have a look at the first test data instance
    classifier = train_classifier(train_data)  # train the classifier
    test_true = [t[1] for t in test_data]   # *get the ground-truth labels from the data
    test_pred = predict_labels([x[0] for x in test_data], classifier)  # *classify the test data to get predicted labels
    final_scores = precision_recall_fscore_support(test_true, test_pred, average='weighted') # evaluate
    print("Done training!")
    print("Precision: %f\nRecall: %f\nF Score:%f" % final_scores[:3])